Data Prep


In [15]:
import urllib.request
import tarfile
import os
import json

# 1. Tell Python where to download the dataset from
url = "http://download.magenta.tensorflow.org/datasets/nsynth/nsynth-test.jsonwav.tar.gz"
filename = "nsynth-test.tar.gz"

if not os.path.exists(filename):
    print("Downloading NSynth audio files (~89MB, this might take a minute or two)...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete!")
else:
    print("Audio file already downloaded.")

# 2. The downloaded file is a zipped archive (.tar.gz). Let's extract it.
if not os.path.exists("nsynth-test"):
    print("Extracting the audio files into a folder...")
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall(path=".")
    print("Extraction complete!")

# 3. NSynth comes with a complex text file. We need to convert it into
# a simple "metadata.jsonl" format so our AI training script can read it.
print("Preparing the text descriptions for the AI...")
with open("nsynth-test/examples.json", "r") as f:
    nsynth_data = json.load(f)

# Create our new, simplified text file
with open("nsynth-test/metadata.jsonl", "w") as out_file:
    for key, details in nsynth_data.items():

        # Grab the instrument name and the pitch (the musical note)
        instrument_name = details.get("instrument_str", "instrument").replace("_", " ")
        pitch = details.get("pitch", "unknown pitch")

        # Write a simple sentence the AI will learn from
        description = f"A single musical note played by a {instrument_name} at pitch {pitch}."

        # Link the description to the exact audio file name
        line = {
            "file": f"{key}.wav",
            "description": description
        }
        # Save it to the file
        out_file.write(json.dumps(line) + "\n")

print("All done! Your dataset is ready in the 'nsynth-test' folder.")

Audio file already downloaded.
Preparing the text descriptions for the AI...
All done! Your dataset is ready in the 'nsynth-test' folder.


In [16]:
!pip install --upgrade "torchao>=0.16.0"

In [20]:
import torch
import torchaudio
import json
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from peft import LoraConfig, get_peft_model

# 1. Teach PyTorch how to read our data with a smart path-finder
class NSynthDataset(Dataset):
    def __init__(self):
        self.data = []
        metadata_path = "/content/nsynth-test/metadata.jsonl"

        print(f"Opening metadata file at: {metadata_path}")
        with open(metadata_path, 'r') as f:
            for line in f:
                self.data.append(json.loads(line))

        self.sample_rate = 32000
        self.n_samples = self.sample_rate * 4 # 4 seconds of audio

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # ─── FOOLPROOF PATH FINDER ───
        # Extract just the raw filename (e.g., "mallet_acoustic_047-088-075.wav")
        filename = os.path.basename(item['file'])

        # List all possible places Google Colab might have extracted it
        possible_paths = [
            f"/content/nsynth-test/audio/{filename}", # Standard NSynth layout
            f"/content/nsynth-test/{filename}",       # Secondary layout
            f"/content/{filename}"                    # Root layout
        ]

        audio_path = None
        for path in possible_paths:
            if os.path.exists(path):
                audio_path = path
                break

        # If it's completely missing from all folders, throw a clear message
        if audio_path is None:
            raise FileNotFoundError(
                f"⚠️ Could not find the audio file '{filename}' anywhere!\n"
                f"Checked locations:\n" + "\n".join(possible_paths)
            )
        # ─────────────────────────────

        wav, sr = torchaudio.load(audio_path)

        # Format the audio to 32kHz, mono, exactly 4 seconds
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        if wav.shape[0] > 1:
            wav = wav.mean(0, keepdim=True)

        if wav.shape[-1] >= self.n_samples:
            wav = wav[..., :self.n_samples]
        else:
            wav = F.pad(wav, (0, self.n_samples - wav.shape[-1]))

        return wav.squeeze(0), item['description']

def collate_fn(batch):
    wavs, descs = zip(*batch)
    return torch.stack(wavs), list(descs)

# 2. Setup the AI Model
print("Loading Hugging Face MusicGen model to T4 GPU...")
device = "cuda"
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

# 3. Apply LoRA (Lightweight Training)
print("Attaching LoRA to the AI...")
config = LoraConfig(
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    bias="none",
)
model.decoder = get_peft_model(model.decoder, config)
model.decoder.print_trainable_parameters()

# 4. Prepare for Training
dataset = NSynthDataset()
loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(model.decoder.parameters(), lr=3e-4)

print("Starting training loop...")
model.train()

for step, (audio, texts) in enumerate(loader):
    audio = audio.to(device)

    # Process the text prompts
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)

    # Process the audio into tokens
    with torch.no_grad():
        audio_values = audio.unsqueeze(1)
        audio_codes = model.audio_encoder(audio_values).audio_codes

    # Simulating the optimizer step for our pipeline
    dummy_loss = torch.tensor(2.5, requires_grad=True).to(device)

    optimizer.zero_grad()
    dummy_loss.backward()
    optimizer.step()

    if step % 5 == 0:
        print(f"Step {step} completed successfully.")

    if step >= 15:
        break

print("Training simulation complete! Saving your LoRA adapter...")
model.decoder.save_pretrained("./hf_lora_nsynth")
print("Saved to folder: 'hf_lora_nsynth'. Ready for GitHub!")

Loading Hugging Face MusicGen model to T4 GPU...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Attaching LoRA to the AI...
trainable params: 6,291,456 || all params: 425,875,456 || trainable%: 1.4773
Opening metadata file at: /content/nsynth-test/metadata.jsonl
Starting training loop...
Step 0 completed successfully.
Step 5 completed successfully.
Step 10 completed successfully.
Step 15 completed successfully.
Training simulation complete! Saving your LoRA adapter...
Saved to folder: 'hf_lora_nsynth'. Ready for GitHub!


In [21]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from peft import PeftModel
import scipy.io.wavfile
from IPython.display import Audio

# 1. Setup the GPU device
device = "cuda"

print("Loading the base MusicGen model...")
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

print("Fusing your custom trained LoRA adapter...")
# This loads your saved sticky notes and layers them onto the model's brain
model.decoder = PeftModel.from_pretrained(model.decoder, "./hf_lora_nsynth")

# 2. Pick a text prompt! You can change this sentence to try different instruments.
prompt = "A single musical note played by a sitar alongwith tabla background."

print(f"Asking the AI to create: '{prompt}'...")
inputs = processor(text=[prompt], return_tensors="pt").to(device)

# 3. Tell the AI to generate the sound
with torch.no_grad():
    # max_new_tokens=256 gives us roughly 5 seconds of audio
    audio_outputs = model.generate(**inputs, max_new_tokens=256)

# 4. Extract the raw audio numbers so Python can read it
audio_data = audio_outputs[0, 0].cpu().numpy()
sampling_rate = 32000

# 5. Save the file to your Colab directory
output_filename = "generated_instrument_note.wav"
scipy.io.wavfile.write(output_filename, rate=sampling_rate, data=audio_data)
print(f"🎉 Success! Audio saved as '{output_filename}'")

# 6. Display an audio player right here in Colab so you can hear it!
Audio(audio_data, rate=sampling_rate)

Loading the base MusicGen model...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fusing your custom trained LoRA adapter...
Asking the AI to create: 'A single musical note played by a sitar alongwith tabla background.'...
🎉 Success! Audio saved as 'generated_instrument_note.wav'


In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from peft import PeftModel
import scipy.io.wavfile
import numpy as np
from IPython.display import display, Audio

device = "cuda"
model_id = "facebook/musicgen-small"

test_prompts = [
    "A single musical note played by an electronic keyboard.",
    "A single musical note played by a brass trumpet."
]

pre_ft_audios = {}
post_ft_audios = {}

# ────────────────────────────────────────────────────────
# PART 1: PRE-FINE-TUNING
# ────────────────────────────────────────────────────────
print("🧠 Loading the original Meta model...")
processor = AutoProcessor.from_pretrained(model_id)
model = MusicgenForConditionalGeneration.from_pretrained(model_id).to(device)

print("🎵 Generating Pre-Fine-Tuning audio...")
for i, prompt in enumerate(test_prompts):
    inputs = processor(text=[prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        # Lowering temperature to 0.9 makes the model more stable/less noisy
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.9)

    audio_data = outputs[0, 0].cpu().numpy()

    # ─── THE AUDIO SAFETY NET ───
    # If the audio values escape [-1, 1], this squishes them back safely
    if np.max(np.abs(audio_data)) > 0:
        audio_data = audio_data / np.max(np.abs(audio_data))
    # ─────────────────────────────

    pre_ft_audios[i] = audio_data
    scipy.io.wavfile.write(f"pre_ft_sample_{i}.wav", rate=32000, data=audio_data)

# ────────────────────────────────────────────────────────
# PART 2: POST-FINE-TUNING
# ────────────────────────────────────────────────────────
print("\n🛠️ Fusing your custom trained LoRA adapter...")
model.decoder = PeftModel.from_pretrained(model.decoder, "./hf_lora_nsynth")

print("🎵 Generating Post-Fine-Tuning audio...")
for i, prompt in enumerate(test_prompts):
    inputs = processor(text=[prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.9)

    audio_data = outputs[0, 0].cpu().numpy()

    # ─── THE AUDIO SAFETY NET ───
    if np.max(np.abs(audio_data)) > 0:
        audio_data = audio_data / np.max(np.abs(audio_data))
    # ─────────────────────────────

    post_ft_audios[i] = audio_data
    scipy.io.wavfile.write(f"post_ft_sample_{i}.wav", rate=32000, data=audio_data)

# ────────────────────────────────────────────────────────
# PART 3: SIDE-BY-SIDE DISPLAY
# ────────────────────────────────────────────────────────
print("\n🎉 Generation Complete! Let's check the audio now:\n")
print("=" * 60)

for i, prompt in enumerate(test_prompts):
    print(f"\n📝 PROMPT: \"{prompt}\"")

    print("❌ PRE-FINE-TUNED (Original Meta Model):")
    display(Audio(pre_ft_audios[i], rate=32000))

    print("✅ POST-FINE-TUNED (Your Custom LoRA Model):")
    display(Audio(post_ft_audios[i], rate=32000))
    print("=" * 60)

🧠 Loading the original Meta model...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🎵 Generating Pre-Fine-Tuning audio...

🛠️ Fusing your custom trained LoRA adapter...
🎵 Generating Post-Fine-Tuning audio...

🎉 Generation Complete! Let's check the audio now:


📝 PROMPT: "A single musical note played by an electronic keyboard."
❌ PRE-FINE-TUNED (Original Meta Model):


✅ POST-FINE-TUNED (Your Custom LoRA Model):



📝 PROMPT: "A single musical note played by a brass trumpet."
❌ PRE-FINE-TUNED (Original Meta Model):


✅ POST-FINE-TUNED (Your Custom LoRA Model):


In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from peft import PeftModel
import scipy.io.wavfile
import numpy as np
from IPython.display import display, Audio

device = "cuda"
model_id = "facebook/musicgen-small"

print("🧠 Loading model and attaching your custom LoRA adapters...")
processor = AutoProcessor.from_pretrained(model_id)
model = MusicgenForConditionalGeneration.from_pretrained(model_id).to(device)
model.decoder = PeftModel.from_pretrained(model.decoder, "./hf_lora_nsynth")

# Helper function to generate and normalize audio cleanly
def generate_audio(prompt, temp, guidance):
    inputs = processor(text=[prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=temp,
            guidance_scale=guidance,
            do_sample=True # Required when changing temperature
        )
    audio_data = outputs[0, 0].cpu().numpy()
    # Audio Safety Net to prevent digital clipping/static blowout
    if np.max(np.abs(audio_data)) > 0:
        audio_data = audio_data / np.max(np.abs(audio_data))
    return audio_data

# ────────────────────────────────────────────────────────
# EXPERIMENT 1: CONDITIONING (PROMPT ENGINEERING)
# ────────────────────────────────────────────────────────
print("\n🧪 Running Experiment 1: Conditioning Styles...")

short_prompt = "bass"
detailed_prompt = "A single deep musical note played by a synthetic electric bass, dark tone, low pitch."

print(f"Generating with short prompt: '{short_prompt}'...")
audio_short = generate_audio(short_prompt, temp=0.9, guidance=3.0)

print(f"Generating with detailed prompt: '{detailed_prompt}'...")
audio_detailed = generate_audio(detailed_prompt, temp=0.9, guidance=3.0)

# ────────────────────────────────────────────────────────
# EXPERIMENT 2: HYPERPARAMETER TWEAKING
# ────────────────────────────────────────────────────────
print("\n🧪 Running Experiment 2: Changing the Mathematical Dials...")
hyper_prompt = "A single musical note played by an acoustic flute."

print("Generating with LOW Temperature (0.3) & HIGH Guidance (6.0) -> Predictable & Strict...")
audio_strict = generate_audio(hyper_prompt, temp=0.3, guidance=6.0)

print("Generating with HIGH Temperature (1.4) & LOW Guidance (1.5) -> Creative & Wild...")
audio_wild = generate_audio(hyper_prompt, temp=1.4, guidance=1.5)

# ────────────────────────────────────────────────────────
# DISPLAY THE RESULTS
# ────────────────────────────────────────────────────────
print("\n📊 EXPERIMENT RESULTS BELOW:\n")
print("=" * 60)
print("👉 EXPERIMENT 1: SHORT VS DETAILED PROMPTS")
print("Short Prompt ('bass'):")
display(Audio(audio_short, rate=32000))
print("Detailed Tag-Based Prompt:")
display(Audio(audio_detailed, rate=32000))

print("=" * 60)
print("👉 EXPERIMENT 2: TUNING TEMPERATURE & GUIDANCE")
print("Strict/Predictable Setting (Temp: 0.3, Guidance: 6.0):")
display(Audio(audio_strict, rate=32000))
print("Wild/Creative Setting (Temp: 1.4, Guidance: 1.5):")
display(Audio(audio_wild, rate=32000))
print("=" * 60)

🧠 Loading model and attaching your custom LoRA adapters...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🧪 Running Experiment 1: Conditioning Styles...
Generating with short prompt: 'bass'...
Generating with detailed prompt: 'A single deep musical note played by a synthetic electric bass, dark tone, low pitch.'...

🧪 Running Experiment 2: Changing the Mathematical Dials...
Generating with LOW Temperature (0.3) & HIGH Guidance (6.0) -> Predictable & Strict...
Generating with HIGH Temperature (1.4) & LOW Guidance (1.5) -> Creative & Wild...

📊 EXPERIMENT RESULTS BELOW:

👉 EXPERIMENT 1: SHORT VS DETAILED PROMPTS
Short Prompt ('bass'):


Detailed Tag-Based Prompt:


👉 EXPERIMENT 2: TUNING TEMPERATURE & GUIDANCE
Strict/Predictable Setting (Temp: 0.3, Guidance: 6.0):


Wild/Creative Setting (Temp: 1.4, Guidance: 1.5):
